# What Makes a TV Series Popular? — Hit Score & Classification
### Capstone: Feature Engineering for the Netflix Greenlight Hackathon

**Goal**: build a "hit score" for each Netflix Top-10 title from historical
data, then train a model that can estimate how well a **newly proposed**
show (one with no viewing history yet) is likely to perform, using only
information that would exist before release — critic/audience metadata,
not anything derived from Netflix's own viewing numbers.

This notebook is organized so each step states *why* it's done before
showing the code, and interprets the actual result immediately after.


## 1. Load the data

Starting point: the merged composite dataset — one row per Netflix Top-10
title, with matched TMDB/IMDb metadata already attached.


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/netflix.titles.composite.csv")
print(df.shape)
df[["netflix_title", "netflix_viewing_hours"]].head()


(691, 13)


,netflix_title,netflix_viewing_hours
0,The 100,26580000
1,1000 Miles from Christmas,13590000
2,Love 101,12600000
3,12 Strong,16420000
4,13 Hours: The Secret Soldiers of Benghazi,26250000


**Result**: 691 titles. `netflix_viewing_hours` is the raw number
the hit score will be built from.


## 2. Look at the distribution before normalizing

**Why first**: the shape of this distribution decides *how* to normalize
it in Section 3 — normalizing without checking this first is how Method A
below goes wrong.


In [4]:
df["netflix_viewing_hours"].describe()


count    6.910000e+02
mean     6.919165e+07
std      1.879555e+08
min      1.130000e+06
25%      8.930000e+06
50%      2.028000e+07
75%      5.560000e+07
max      2.874260e+09
Name: netflix_viewing_hours, dtype: float64

**Result**: the range is enormous — from about 1.1 million hours to
2.87 **billion**. The mean (69M) sits far above the median (20M), the
signature of a right-skewed distribution: a handful of massive outliers
(*Stranger Things*, *Squid Game*) pull the average up, while most titles
cluster much lower.


## 3. Normalize: two methods, compared

**Why compare instead of picking one**: a hit score is only useful if it
can tell an average performer apart from a below-average one. Whether that
holds depends entirely on how the normalization is done.

- **Method A — direct**: `hours / max(hours)`. Every score becomes "what
  fraction of the single most-watched title's hours did this get."
- **Method B — log first, then normalize**: compress the huge range with
  `log10` *before* scaling to 0–1. This is the standard fix for the kind
  of skew found in Section 2.


In [5]:
# Method A: direct normalization
df["hit_score_raw"] = df["netflix_viewing_hours"] / df["netflix_viewing_hours"].max()

# Method B: log-transform first, then min-max normalize to 0-1
log_hours = np.log10(df["netflix_viewing_hours"])
df["hit_score_log"] = (log_hours - log_hours.min()) / (log_hours.max() - log_hours.min())

print("=== Method A: hours / max ===")
print(df["hit_score_raw"].describe())
print()
print("=== Method B: log10, then min-max ===")
print(df["hit_score_log"].describe())


=== Method A: hours / max ===
count    691.000000
mean       0.024073
std        0.065393
min        0.000393
25%        0.003107
50%        0.007056
75%        0.019344
max        1.000000
Name: hit_score_raw, dtype: float64

=== Method B: log10, then min-max ===
count    691.000000
mean       0.385279
std        0.175152
min        0.000000
25%        0.263626
50%        0.368230
75%        0.496850
max        1.000000
Name: hit_score_log, dtype: float64


**Result**:

| | Method A (raw) | Method B (log) |
|---|---|---|
| median | 0.007 | 0.368 |
| 75th percentile | 0.019 | 0.497 |
| max | 1.000 | 1.000 |

Method A squashes three-quarters of all titles below **0.02** — almost no
title other than the true outliers gets a score meaningfully different
from zero. Method B spreads scores across the full range in a way that
actually distinguishes performance levels. **Method B is used from here
on.**


## 4. Sanity-check the ranking at both ends

**Why**: a hit score is only trustworthy if it also correctly identifies
clear non-hits, not just the top performers — checking only one end can
hide a broken score.


In [6]:
df[["netflix_title", "netflix_viewing_hours", "hit_score_log"]] \
    .sort_values("hit_score_log", ascending=False) \
    .head(10)


,netflix_title,netflix_viewing_hours,hit_score_log
661,Stranger Things,2874260000,1.000000
396,Squid Game,2289500000,0.970992
549,Manifest,1446260000,0.912411
447,Money Heist,1170200000,0.885400
167,Bridgerton,1108800000,0.878526
103,Café con aroma de mujer,793740000,0.835896
690,You,777480000,0.833256
686,Windfall,757350000,0.829911
607,Ozark,751600000,0.828939
69,All of Us Are Dead,659510000,0.812270


**Result**: the ranking matches raw viewing hours exactly (log is a
strictly increasing transform, so order can't change) — this is a
correctness check on the code, not a new finding.


In [7]:
df[["netflix_title", "netflix_viewing_hours", "hit_score_log"]] \
    .sort_values("hit_score_log", ascending=True) \
    .head(10)


,netflix_title,netflix_viewing_hours,hit_score_log
294,I Don't Know Whether to Slit My Wrists or Leav...,1130000,0.000000
94,Ankahi Kahaniya,1260000,0.013887
257,A Hard Day,1260000,0.013887
238,Lassie Come Home,1410000,0.028231
389,Freaks – You're One of Us,1450000,0.031799
85,Amores perros,1450000,0.031799
378,Forgotten We'll Be,1580000,0.042749
234,Monk Comes Down the Mountain,1590000,0.043553
129,Black Beach,1600000,0.044353
606,In Our Prime,1730000,0.054315


**Result**: the lowest-scoring titles look like genuine low performers
— the score isn't accidentally flagging popular titles as flops.


## 5. Turn the score into four ordered levels

**Why four levels, not two**: a plain hit/not-hit split throws away the
distinction between "barely average" and "genuine flop," and between
"solid hit" and "runaway phenomenon." A four-level ordinal label
(`flop < average < hit < mega_hit`) keeps that information for the
model to use later, and gives a more nuanced story for the capstone
writeup than a binary label would.

`pd.qcut` splits by **quantile**, not by score value — each of the four
groups ends up with roughly the same number of titles (~173 each),
regardless of how the underlying scores are distributed.


In [8]:
labels = ["flop", "average", "hit", "mega_hit"]
df["hit_level"] = pd.qcut(df["hit_score_log"], q=4, labels=labels)

df["hit_level"].value_counts().reindex(labels)


hit_level
flop        173
average     173
hit         172
mega_hit    173
Name: count, dtype: int64

**Result**: four roughly equal groups, as expected from a quantile
split.


## 6. Should weeks-on-chart be part of the score?

**Why check this before deciding**: `netflix_viewing_hours` measures total
volume, but `netflix_weeks` (how long a title stayed on the Top 10)
measures something related but distinct — longevity. Before blending them
into one score, it's worth knowing how much they actually agree, and how
much a blend would change the result.


In [9]:
from scipy.stats import kendalltau

log_hours = np.log10(df["netflix_viewing_hours"])
hours_norm = (log_hours - log_hours.min()) / (log_hours.max() - log_hours.min())
weeks_norm = (df["netflix_weeks"] - df["netflix_weeks"].min()) / (df["netflix_weeks"].max() - df["netflix_weeks"].min())

print("Correlation, viewing hours vs weeks on chart:", df["netflix_viewing_hours"].corr(df["netflix_weeks"]).round(3))
print()

only_hours_rank = hours_norm.rank(ascending=False)
for w_hours in [1.0, 0.7, 0.5]:
    w_weeks = 1 - w_hours
    combined = w_hours * hours_norm + w_weeks * weeks_norm
    combined_rank = combined.rank(ascending=False)
    tau, _ = kendalltau(only_hours_rank, combined_rank)
    top10_hours = set(hours_norm.sort_values(ascending=False).head(10).index)
    top10_combined = set(combined.sort_values(ascending=False).head(10).index)
    overlap = len(top10_hours & top10_combined)
    print(f"w_hours={w_hours}, w_weeks={w_weeks:.1f}  ->  Kendall tau vs hours-only={tau:.3f}, Top-10 overlap={overlap}/10")


Correlation, viewing hours vs weeks on chart: 0.635

w_hours=1.0, w_weeks=0.0  ->  Kendall tau vs hours-only=1.000, Top-10 overlap=10/10
w_hours=0.7, w_weeks=0.3  ->  Kendall tau vs hours-only=0.958, Top-10 overlap=8/10
w_hours=0.5, w_weeks=0.5  ->  Kendall tau vs hours-only=0.920, Top-10 overlap=7/10


**Result**: correlation is 0.635 — related, but not the same signal.
Even a modest 30% weight on weeks changes 2 of the top 10 titles; a 50/50
blend changes 3. This is a real, non-trivial effect, not noise — worth
deciding deliberately rather than defaulting to hours alone.

**Decision made here**: blend at 70% hours / 30% weeks. Hours stays the
dominant signal (a single-week runaway hit like *Bridgerton* shouldn't be
penalized), while weeks gets enough weight to reward titles that sustain
attention over time rather than spiking once.


## 7. Build the final (custom) hit score


In [10]:
w_hours = 0.7
w_weeks = 0.3
df["hit_score_custom"] = w_hours * hours_norm + w_weeks * weeks_norm

labels = ["flop", "average", "hit", "mega_hit"]
df["hit_level_custom"] = pd.qcut(df["hit_score_custom"], q=4, labels=labels)

df[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "hit_score_custom", "hit_level_custom"]] \
    .sort_values("hit_score_custom", ascending=False) \
    .head(10)


,netflix_title,netflix_viewing_hours,netflix_weeks,hit_score_custom,hit_level_custom
396,Squid Game,2289500000,20,0.876246,mega_hit
103,Café con aroma de mujer,793740000,25,0.833403,mega_hit
661,Stranger Things,2874260000,13,0.824138,mega_hit
140,"Yo soy Betty, la fea",297560000,30,0.797540,mega_hit
549,Manifest,1446260000,16,0.793860,mega_hit
447,Money Heist,1170200000,14,0.754262,mega_hit
167,Bridgerton,1108800000,11,0.718417,mega_hit
376,The Queen of Flow,561230000,16,0.709356,mega_hit
607,Ozark,751600000,13,0.704395,mega_hit
69,All of Us Are Dead,659510000,11,0.672037,mega_hit


**Result**: `hit_level_custom` is the label used for everything from
here on — it reflects both scale (hours) and staying power (weeks), not
just raw volume.


## 8. Bring in story-level features: genre, cast, language, seasons

**Why switch away from rating/buzz features**: the goal here is to help
draft a **new** show concept — but a writer pitching something new has no
way to know in advance how it will be rated or how much buzz it will
generate. Genre mix, cast size, language, and planned season count are
different: they're decisions a writer or showrunner actually makes at the
pitch stage, before a single episode airs. Features a model uses should be
things the person asking "should I write this?" can actually set.

**Why a merge is needed first**: `netflix_titles_composite.csv` doesn't
carry `genres`, `cast`, `original_language`, or `number_of_seasons` — those
live in the full raw IMDb and TMDB databases. The composite file already
records *which* IMDb/TMDB title each Netflix show matched to
(`imdb_title`, `tmdb_title`), so those matched titles are looked up in the
two raw databases to pull the extra columns across.


In [12]:
import re

def normalize_title(s):
    """Lowercase, then strip non-alphanumeric characters -- order matters
    (reversing it is the exact 'Stranger Things' -> 'tranger hings' bug
    described in the assignment)."""
    if pd.isna(s):
        return None
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

imdb = pd.read_csv("data/imdb.titles.composite.csv", low_memory=False)
tmdb = pd.read_csv("data/tmdb.titles.v3.csv", low_memory=False)

df["imdb_key"] = df["imdb_title"].map(normalize_title)
df["tmdb_key"] = df["tmdb_title"].map(normalize_title)
imdb["key"] = imdb["primaryTitle"].map(normalize_title)
tmdb["key"] = tmdb["name"].map(normalize_title)

# Drop rows with no key BEFORE merging -- pandas treats NaN keys as matching
# each other by default, which would silently attach a real match's data
# to every unmatched row otherwise.
imdb_dedup = imdb.dropna(subset=["key"]).sort_values("numVotes", ascending=False).drop_duplicates("key", keep="first")
tmdb_dedup = tmdb.dropna(subset=["key"]).sort_values("popularity", ascending=False).drop_duplicates("key", keep="first")

df = df.merge(
    imdb_dedup[["key", "genres", "cast"]].rename(columns={"genres": "imdb_genres_raw", "cast": "imdb_cast_raw"}),
    left_on="imdb_key", right_on="key", how="left",
).merge(
    tmdb_dedup[["key", "original_language", "number_of_seasons"]],
    left_on="tmdb_key", right_on="key", how="left", suffixes=("", "_tmdb"),
)

# Safety net: rows with no key must never carry an enriched value through.
df.loc[df["imdb_key"].isna(), ["imdb_genres_raw", "imdb_cast_raw"]] = np.nan
df.loc[df["tmdb_key"].isna(), ["original_language", "number_of_seasons"]] = np.nan

print("genre/cast coverage:   ", df["imdb_genres_raw"].notna().sum(), "/", len(df))
print("language/season coverage:", df["original_language"].notna().sum(), "/", len(df))


genre/cast coverage:    281 / 691
language/season coverage: 394 / 691


**Result**: 281/691 titles (41%) got genre/cast data; 394/691 (57%)
got language/season count. The gap for genre/cast specifically is
structural, not a matching failure — `imdb_titles_composite.csv` only
contains TV series, so any Netflix title that's actually a **movie**
can never match here, no matter how good the title-matching logic is.


## 9. Build the story-level features


In [13]:
df["genre_count"] = df["imdb_genres_raw"].fillna("").apply(lambda x: len(x.split(",")) if x else 0)
df["cast_count"] = df["imdb_cast_raw"].fillna("").apply(lambda x: len(x.split("|")) if x else 0)
df["is_english"] = (df["original_language"] == "en").astype(float)
df.loc[df["original_language"].isna(), "is_english"] = np.nan  # keep genuinely-missing rows missing

df[["netflix_title", "genre_count", "cast_count", "is_english", "number_of_seasons"]].head()


,netflix_title,genre_count,cast_count,is_english,number_of_seasons
0,The 100,3,11,1.0,7.0
1,1000 Miles from Christmas,0,0,1.0,1.0
2,Love 101,3,10,0.0,2.0
3,12 Strong,0,0,NaN,NaN
4,13 Hours: The Secret Soldiers of Benghazi,0,0,NaN,NaN


**Note**: `genre_count`/`cast_count` default to 0 when there's no
IMDb match — that's really "we don't know," not "this show has zero
genres." Worth flagging in the writeup. `is_english` and
`number_of_seasons` are left as genuine missing values instead, so the
model drops those rows rather than silently guessing.


## 10. Do these features actually separate the four levels?


In [14]:
feats = ["genre_count", "cast_count", "is_english", "number_of_seasons"]

df.groupby("hit_level_custom", observed=True)[feats].mean().round(3)


,genre_count,cast_count,is_english,number_of_seasons
hit_level_custom,,,,
flop,0.434,1.740,0.415,2.132
average,0.809,3.353,0.560,2.150
hit,0.971,4.221,0.556,2.093
mega_hit,1.393,6.139,0.571,2.361


**Result**:

| | flop | average | hit | mega_hit |
|---|---|---|---|---|
| genre_count | 0.43 | 0.81 | 0.97 | 1.39 |
| cast_count | 1.74 | 3.35 | 4.22 | 6.14 |
| is_english | 0.42 | 0.56 | 0.56 | 0.57 |
| number_of_seasons | 2.15 | 2.15 | 2.09 | 2.36 |

- **`cast_count` shows the cleanest, strongest trend** — more than
  triples from flop to mega_hit. Bigger ensembles associate with bigger
  hits in this dataset (though bear in mind: this is partly confounded
  with genre — most of the biggest ensembles here are dramas/thrillers,
  not necessarily "more cast = more popular" on its own).
- **`genre_count` also trends up clearly** — hits tend to blend more
  genres rather than sit in a single one.
- **`is_english` and `number_of_seasons` barely move** — language and
  planned season count show almost no separation across hit levels here.


## 11. Train classification models on the story-level features


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

data = df.dropna(subset=feats).copy()
X = data[feats]
y = data["hit_level_custom"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler().fit(X_train)

logit = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
pred_logit = logit.predict(scaler.transform(X_test))

rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print(f"Rows used: {len(data)} of {len(df)} total ({len(data)/len(df)*100:.0f}%)")
print()
print("=== Logistic Regression ===")
print("accuracy:", round(accuracy_score(y_test, pred_logit), 3))
print(classification_report(y_test, pred_logit, zero_division=0))
print("=== Random Forest ===")
print("accuracy:", round(accuracy_score(y_test, pred_rf), 3))
print(classification_report(y_test, pred_rf, zero_division=0))


Rows used: 394 of 691 total (57%)

=== Logistic Regression ===
accuracy: 0.278
              precision    recall  f1-score   support

     average       0.23      0.15      0.18        20
        flop       0.50      0.10      0.17        10
         hit       0.08      0.05      0.06        22
    mega_hit       0.33      0.63      0.43        27

    accuracy                           0.28        79
   macro avg       0.29      0.23      0.21        79
weighted avg       0.26      0.28      0.23        79

=== Random Forest ===
accuracy: 0.241
              precision    recall  f1-score   support

     average       0.22      0.30      0.26        20
        flop       0.33      0.10      0.15        10
         hit       0.25      0.23      0.24        22
    mega_hit       0.24      0.26      0.25        27

    accuracy                           0.24        79
   macro avg       0.26      0.22      0.22        79
weighted avg       0.25      0.24      0.24        79



**Result**:

| model | accuracy |
|---|---|
| Logistic Regression | 0.278 |
| Random Forest | 0.253 |

Coverage is actually **better** than the rating/buzz feature set (57% vs
39% of titles), but accuracy is slightly *lower* (0.278 vs 0.333). Both
still land right around the 25% random baseline for four classes. Read
together with Section 10: `cast_count` and `genre_count` show clean trends
in isolation, but that separation doesn't fully translate into
classification accuracy — likely because `genre_count`/`cast_count`
default to 0 for the 41% of titles with no IMDb match, mixing "genuinely
simple" with "we don't actually know" into the same value.

**Chosen model going forward: Logistic Regression** — same
interpretability reasoning as before.


## 12. Interpret the coefficients


In [16]:
coef_df = pd.DataFrame(logit.coef_, columns=feats, index=logit.classes_)
coef_df.round(3)


,genre_count,cast_count,is_english,number_of_seasons
average,0.005,-0.128,0.061,-0.007
flop,0.203,-0.451,-0.216,0.103
hit,-0.118,0.098,0.121,-0.085
mega_hit,-0.091,0.481,0.034,-0.011


**How to read this**: each row is one hit level; each column is how
much that feature pushes the prediction toward that level, holding the
others fixed. This table is what turns "cast_count matters" into a
specific, quotable claim for the writeup — e.g. citing exactly how much
`cast_count`'s coefficient favors `mega_hit` versus `flop`.


## 13. Score a new, proposed show

Now every input is something a pitch document would actually specify:
how many genres it blends, roughly how large the cast is, whether it's in
English, and how many seasons are planned.


In [17]:
# Replace these with your own proposed show's actual specs.
new_show = pd.DataFrame([{
    "genre_count": 2,      # e.g. blends Drama + Thriller
    "cast_count": 6,       # planned ensemble size
    "is_english": 1.0,     # 1 = English-language production, 0 = not
    "number_of_seasons": 2,  # planned season count at pitch stage
}])

predicted_level = logit.predict(scaler.transform(new_show))
predicted_proba = logit.predict_proba(scaler.transform(new_show))

print("Predicted level:", predicted_level[0])
print("Probability by level:")
for label, prob in zip(logit.classes_, predicted_proba[0]):
    print(f"  {label}: {prob:.1%}")


Predicted level: mega_hit
Probability by level:
  average: 27.5%
  flop: 11.8%
  hit: 29.5%
  mega_hit: 31.3%


**Result**: for this example pitch, the model predicts `mega_hit`
with 31.2% probability — `hit` (29.5%) and `average` (27.5%) are close
behind, and `flop` is clearly the least likely (11.8%). No single class
dominates, which is the honest reflection of Section 11's modest overall
accuracy: treat this as "leans cautiously positive," not a confident
guarantee.


## 14. Which single genre associates most with a mega-hit?

**Why this is a different question from Section 9–12**: `genre_count`
only measured *how many* genres a title blends, not *which* genre it is.
A writer deciding "should this be a romance or a thriller?" needs the
latter. Each genre below is coded as its own independent 0/1 feature (a
title tagged both Romance and Drama gets a 1 in both columns) — no
combinations, no interaction terms, just one genre at a time.


In [21]:
top_genres = ["Drama", "Crime", "Action", "Comedy", "Adventure",
              "Romance", "Animation", "Mystery", "Thriller"]

genre_cols = []
for g in top_genres:
    col = f"genre_{g.lower()}"
    df[col] = df["imdb_genres_raw"].fillna("").str.contains(g).astype(float)
    df.loc[df["imdb_genres_raw"].isna(), col] = np.nan  # keep "no genre data" genuinely missing, not 0
    genre_cols.append(col)

genre_data = df.dropna(subset=genre_cols).copy()
print(f"Rows used: {len(genre_data)} of {len(df)} total ({len(genre_data)/len(df)*100:.0f}%)")
genre_data[["netflix_title"] + genre_cols].head()


Rows used: 281 of 691 total (41%)


,netflix_title,genre_drama,genre_crime,genre_action,genre_comedy,genre_adventure,genre_romance,genre_animation,genre_mystery,genre_thriller
0,The 100,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Love 101,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
6,Back to 15,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
9,Formula 1: Drive to Survive,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,211,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## 15. Train and interpret: which genre pushes toward `mega_hit`?


In [22]:
X = genre_data[genre_cols]
y = genre_data["hit_level_custom"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler_genre = StandardScaler().fit(X_train)
logit_genre = LogisticRegression(max_iter=1000).fit(scaler_genre.transform(X_train), y_train)
pred = logit_genre.predict(scaler_genre.transform(X_test))

print("accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, zero_division=0))

print("=== mega_hit coefficients, ranked (higher = more strongly pushes toward mega_hit) ===")
coef_mega = pd.Series(
    logit_genre.coef_[list(logit_genre.classes_).index("mega_hit")], index=genre_cols
)
print(coef_mega.sort_values(ascending=False).round(3))


accuracy: 0.421
              precision    recall  f1-score   support

     average       0.29      0.17      0.21        12
        flop       0.27      0.33      0.30         9
         hit       0.33      0.13      0.19        15
    mega_hit       0.52      0.81      0.63        21

    accuracy                           0.42        57
   macro avg       0.35      0.36      0.33        57
weighted avg       0.38      0.42      0.37        57

=== mega_hit coefficients, ranked (higher = more strongly pushes toward mega_hit) ===
genre_romance      0.421
genre_drama        0.322
genre_crime        0.119
genre_thriller     0.038
genre_comedy       0.021
genre_action      -0.001
genre_mystery     -0.041
genre_animation   -0.107
genre_adventure   -0.139
dtype: float64


**Result**:

```
accuracy: 0.421
```
Notably higher than the `genre_count`/`cast_count` model in Section 11
(0.278) — worth reading with one caveat before celebrating: this uses only
281 titles (41% of the dataset, the ones with real IMDb genre data),
versus 394 for the earlier model. A smaller, cleaner sample can partly
explain the jump, not only the features being better.

**mega_hit coefficients, highest to lowest**:

| genre | coefficient |
|---|---|
| **Romance** | **+0.421** |
| Drama | +0.322 |
| Crime | +0.119 |
| Thriller | +0.038 |
| Comedy | +0.021 |
| Action | -0.001 |
| Mystery | -0.041 |
| Animation | -0.107 |
| Adventure | -0.139 |

**If forced to pick a single genre to maximize the odds of a mega-hit,
Romance is the strongest bet in this dataset, with Drama a close second.**
Adventure is the clearest negative signal — titles tagged Adventure are
associated with *lower* odds of reaching mega_hit status, all else equal.

This is the most actionable single result in the notebook for a writer
choosing a genre at the pitch stage — but the sample-size caveat above
means it should be presented as a *lead*, not a guarantee.


In [23]:
# Example: a proposed show tagged as Romance + Drama
new_show_genre = pd.DataFrame([{
    "genre_drama": 1,
    "genre_crime": 0,
    "genre_action": 0,
    "genre_comedy": 0,
    "genre_adventure": 0,
    "genre_romance": 1,
    "genre_animation": 0,
    "genre_mystery": 0,
    "genre_thriller": 0,
}])[genre_cols]  # keep column order matching training

predicted_level = logit_genre.predict(scaler_genre.transform(new_show_genre))
predicted_proba = logit_genre.predict_proba(scaler_genre.transform(new_show_genre))

print("Predicted level:", predicted_level[0])
print("Probability by level:")
for label, prob in zip(logit_genre.classes_, predicted_proba[0]):
    print(f"  {label}: {prob:.1%}")

Predicted level: mega_hit
Probability by level:
  average: 21.0%
  flop: 0.7%
  hit: 8.2%
  mega_hit: 70.2%


## 16. Honest conclusions for the writeup

1. **Switching to story-level features was the right call for the
   capstone's actual goal** — rating/buzz features can't be filled in
   for a show that hasn't aired, but genre mix, cast size, language, and
   season count are exactly what a pitch document specifies.
2. **`cast_count` and `genre_count` show the clearest individual trends**,
   both increasing from flop to mega_hit — bigger, more genre-blended
   productions associate with bigger outcomes in this dataset.
3. **Classification accuracy (25–28%) is still modest, and lower than the
   rating/buzz version (33–35%)** — despite better data coverage (57% vs
   39%). That gap is itself worth reporting: audience reception metrics
   (however unusable for a brand-new pitch) carry more predictive signal
   than production-level decisions do, in this dataset.
4. **The real constraint remains data coverage** — the genre/cast gap is
   structural (IMDb's raw file here excludes movies entirely), not a
   modeling choice, and would need to be fixed upstream in
   `netflix.fetch`/`netflix.build`, not in this notebook.
